# Inference & Evaluation - Response Model

Evaluate fine-tuned response model menggunakan:
- LLM-as-Judge (Gemini, skor 0-2 Rompis 2025)
- Reference accuracy
- Hallucination detection
- BERTScore F1

**Prerequisites:**
- Fine-tuned model served via vLLM / OpenAI-compatible API
- Validation CSV dari step 06a
- Gemini API key untuk LLM Judge

## Setup

In [ ]:
import os, sys, json
from dotenv import load_dotenv
from openai import OpenAI

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

load_dotenv(os.path.join(ROOT_DIR, '.env'), override=True)

# Model server config
openai_api_base = "http://localhost:8000/v1"
openai_api_key = os.getenv('OPENAI_API_KEY', 'not-needed')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')

print(f"API Base: {openai_api_base}")
print(f"Gemini: {'set' if GEMINI_API_KEY else 'NOT SET'}")

In [ ]:
client = OpenAI(api_key=openai_api_key, base_url=openai_api_base)

try:
    models = client.models.list()
    print(f"Available models: {[m.id for m in models.data]}")
except Exception as e:
    print(f"Cannot connect: {e}")

MODEL_NAME = "Qwen/Qwen3-4b"  # Replace with fine-tuned model name

## Load test data

In [ ]:
import pandas as pd

TEST_DATA_PATH = os.path.join(ROOT_DIR, 'finetuning', 'response_model', 'data', 'validation_data.csv')
df_test = pd.read_csv(TEST_DATA_PATH)
print(f"Test samples: {len(df_test)}")
df_test.head()

### Sanity check - single inference

In [ ]:
import time
from finetuning.response_model.evaluate import SYSTEM_INSTRUCTION

sample = df_test.iloc[0]
user_msg = (
    f"<INPUT>\n<CONTEXT>\n{sample['context']}\n</CONTEXT>\n"
    f"<QUESTION>\n{sample['question']}\n</QUESTION>\n</INPUT>"
)

start = time.time()
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": user_msg},
    ],
    temperature=0.3,
    max_tokens=1024,
)
elapsed = (time.time() - start) * 1000
predicted = response.choices[0].message.content.strip()

print(f"Q: {sample['question']}")
print(f"\nExpected: {sample['response'][:200]}...")
print(f"\nPredicted: {predicted[:200]}...")
print(f"\nInference: {elapsed:.0f}ms")

## Batch Evaluation

In [ ]:
from finetuning.response_model.evaluate import evaluate

metrics = evaluate(
    test_data_path=TEST_DATA_PATH,
    api_base=openai_api_base,
    api_key=openai_api_key,
    model_name=MODEL_NAME,
    judge_api_key=GEMINI_API_KEY,
)

## Results Summary

In [ ]:
metrics_path = os.path.join(ROOT_DIR, 'finetuning', 'response_model', 'data', 'evaluation_results.json')
with open(metrics_path, 'r') as f:
    metrics = json.load(f)

targets = {
    'llm_judge_avg': ('>=', 1.30),
    'reference_accuracy': ('>=', 0.90),
    'hallucination_rate': ('<=', 0.10),
    'bert_score_f1_avg': ('>=', 0.85),
}

print("=" * 60)
print("EVALUATION RESULTS vs DoD TARGETS")
print("=" * 60)
for metric, (op, target) in targets.items():
    actual = metrics.get(metric, 0)
    if op == '>=':
        passed = actual >= target
        fmt = f"{actual:.2f} (target {op} {target:.2f})"
    else:
        passed = actual <= target
        fmt = f"{actual:.2f} (target {op} {target:.2f})"
    status = 'PASS' if passed else 'FAIL'
    print(f"  {metric:25s}: {fmt} [{status}]")

print(f"\n  {'avg_inference_ms':25s}: {metrics.get('avg_inference_ms', 0):.0f}ms")